In [ ]:
library(dplyr)

df <- read.csv("KaggleV2-May-2016.csv")
dim(df)
names(df)

[1] 110527     14

[1] "PatientId"      "AppointmentID"  "Gender"         "ScheduledDay"  
 [5] "AppointmentDay" "Age"            "Neighbourhood"  "Scholarship"   
 [9] "Hipertension"   "Diabetes"       "Alcoholism"     "Handcap"       
[13] "SMS_received"   "No.show"

In [ ]:
df <- df %>%
  mutate(
    sched_date = as.Date(substr(ScheduledDay, 1, 10)),
    appt_date  = as.Date(substr(AppointmentDay, 1, 10)),
    lead_days  = as.numeric(appt_date - sched_date),
    no_show    = ifelse(No.show == "Yes", 1, 0)
  ) %>%
  filter(Age >= 0, lead_days >= 0)

nrow(df)

[1] 110521

In [ ]:
grp <- df %>%
  group_by(SMS_received) %>%
  summarise(n = n(), no_shows = sum(no_show))

grp

prop.test(x = grp$no_shows, n = grp$n)

SMS_received,n,no_shows
<int>,<int>,<dbl>
0,75039,12530
1,35482,9784



	2-sample test for equality of proportions with continuity correction

data:  grp$no_shows out of grp$n
X-squared = 1768, df = 1, p-value < 2.2e-16
alternative hypothesis: two.sided
95 percent confidence interval:
 -0.1141476 -0.1033837
sample estimates:
   prop 1    prop 2 
0.1669798 0.2757454 


In [ ]:
df3 <- df %>% filter(lead_days >= 3)
nrow(df3)

model <- glm(no_show ~ SMS_received + lead_days + Age + Gender +
               Scholarship + Hipertension + Diabetes + Alcoholism,
             data = df3, family = binomial)

summary(model)
round(exp(coef(model)), 3)

[1] 60021


Call:
glm(formula = no_show ~ SMS_received + lead_days + Age + Gender + 
    Scholarship + Hipertension + Diabetes + Alcoholism, family = binomial, 
    data = df3)

Coefficients:
               Estimate Std. Error z value Pr(>|z|)    
(Intercept)  -0.5092391  0.0241996 -21.043  < 2e-16 ***
SMS_received -0.2602106  0.0182429 -14.264  < 2e-16 ***
lead_days     0.0069362  0.0005283  13.131  < 2e-16 ***
Age          -0.0096951  0.0004708 -20.595  < 2e-16 ***
GenderM      -0.0355183  0.0195633  -1.816  0.06944 .  
Scholarship   0.2719838  0.0296404   9.176  < 2e-16 ***
Hipertension -0.0769547  0.0290830  -2.646  0.00814 ** 
Diabetes      0.1243118  0.0411354   3.022  0.00251 ** 
Alcoholism    0.4322633  0.0559874   7.721 1.16e-14 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 72988  on 60020  degrees of freedom
Residual deviance: 71889  on 60012  degrees of freedom
AIC: 71907

Number of F

(Intercept) SMS_received    lead_days          Age      GenderM  Scholarship 
       0.601        0.771        1.007        0.990        0.965        1.313 
Hipertension     Diabetes   Alcoholism 
       0.926        1.132        1.541

In [ ]:
round(exp(confint.default(model)["SMS_received", ]), 3)

2.5 % 97.5 % 
 0.744  0.799